# Tutorial 5: Text Analysis and Sentiment Classification

Welcome to the fifth tutorial in our statistical learning series! In
this notebook, we’ll explore techniques for analyzing text data and
building sentiment classification models.

## Learning Objectives

By the end of this tutorial, you’ll be able to: - Preprocess text data
for analysis - Convert text to numerical features using different
techniques - Build and evaluate sentiment classification models -
Interpret model outputs to understand important features - Apply these
techniques to real-world text data

## 1. Setup and Introduction

Let’s begin by importing the necessary libraries.

``` python
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
from collections import Counter

# NLP libraries
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from wordcloud import WordCloud

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.pipeline import Pipeline

# Import our sentiment analysis utilities
from statistics_lessons.projects.data_loaders import load_newsgroup_sentiment
from statistics_lessons.projects.sentiment_analysis import sentiment_classification

# Download necessary NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
```

## 2. Text Preprocessing Fundamentals

Text preprocessing is crucial for effective analysis. Let’s explore the
key preprocessing steps using a simple example.

``` python
# Sample text data
sample_texts = [
    "This movie was absolutely amazing! I loved every minute of it.",
    "The acting was terrible and the plot made no sense. Waste of time!",
    "It was OK, nothing special but not terrible either.",
    "BEST. FILM. EVER!!! I can't wait to see it again! :)"
]

# Let's create a function to preprocess text
def preprocess_text(text, remove_stopwords=True, stem=False, lemmatize=False):
    """
    Preprocess text by performing the following steps:
    1. Convert to lowercase
    2. Remove punctuation
    3. Remove numbers
    4. Remove stopwords (optional)
    5. Stem or lemmatize words (optional)
    
    Args:
        text (str): Input text to preprocess
        remove_stopwords (bool): Whether to remove stopwords
        stem (bool): Whether to stem words
        lemmatize (bool): Whether to lemmatize words
    
    Returns:
        str: Preprocessed text
    """
    # Convert to lowercase
    text = text.lower()
    
    # Remove punctuation
    text = re.sub(f'[{string.punctuation}]', ' ', text)
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords
    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        tokens = [token for token in tokens if token not in stop_words]
    
    # Stem words
    if stem:
        stemmer = PorterStemmer()
        tokens = [stemmer.stem(token) for token in tokens]
    
    # Lemmatize words
    if lemmatize:
        lemmatizer = WordNetLemmatizer()
        tokens = [lemmatizer.lemmatize(token) for token in tokens]
    
    # Join tokens back into a string
    processed_text = ' '.join(tokens)
    
    return processed_text

# Apply preprocessing to our sample texts
preprocessed_texts = []
for text in sample_texts:
    preprocessed = preprocess_text(text, remove_stopwords=True, lemmatize=True)
    preprocessed_texts.append(preprocessed)
    print(f"Original: {text}")
    print(f"Preprocessed: {preprocessed}\n")
```

### 2.1 Understanding Text Preprocessing Steps

Let’s examine how each preprocessing step affects our text data.

``` python
# Choose one example text
example_text = sample_texts[0]

# Original
print(f"Original: {example_text}")

# Lowercase
lowercase_text = example_text.lower()
print(f"Lowercase: {lowercase_text}")

# Remove punctuation
no_punct_text = re.sub(f'[{string.punctuation}]', ' ', lowercase_text)
print(f"No punctuation: {no_punct_text}")

# Tokenize
tokens = word_tokenize(no_punct_text)
print(f"Tokens: {tokens}")

# Remove stopwords
stop_words = set(stopwords.words('english'))
filtered_tokens = [token for token in tokens if token not in stop_words]
print(f"Without stopwords: {filtered_tokens}")

# Stemming
stemmer = PorterStemmer()
stemmed_tokens = [stemmer.stem(token) for token in filtered_tokens]
print(f"Stemmed: {stemmed_tokens}")

# Lemmatization
lemmatizer = WordNetLemmatizer()
lemmatized_tokens = [lemmatizer.lemmatize(token) for token in filtered_tokens]
print(f"Lemmatized: {lemmatized_tokens}")
```

### Interactive Exercise 1: Effect of Preprocessing Choices

🔍 **Explore how different preprocessing choices affect the text data:**

1.  How does stopword removal change the meaning of the texts?
2.  What are the key differences between stemming and lemmatization?
3.  What preprocessing steps would you choose for sentiment analysis and
    why?
4.  How might preprocessing choices vary depending on the text analysis
    task?

<details>
<summary>
Click for answers
</summary>

1.  Stopword removal effects:
    -   Removes common words like “the”, “was”, “and”, “of”, etc. that
        typically don’t carry much meaning
    -   Reduces text length significantly
    -   May change subtle meanings but generally preserves
        sentiment-bearing words
    -   Helpful for bag-of-words models to focus on meaningful terms
2.  Stemming vs. lemmatization:
    -   Stemming: Cuts off word endings based on heuristic rules (faster
        but cruder)
        -   Example: “running”, “runs”, “runner” → “run”
        -   Often produces non-words like “movi” for “movie”
    -   Lemmatization: Converts words to their dictionary form (slower
        but more accurate)
        -   Example: “better” → “good”, “running” → “run”
        -   Preserves meaning better but requires understanding of word
            context
    -   Stemming is faster but less accurate; lemmatization is slower
        but preserves meaning better
3.  Preprocessing for sentiment analysis:
    -   Lowercase: Important to treat “Good” and “good” as the same word
    -   Punctuation removal: May want to keep some punctuation like “!”
        or “?” as they can indicate emotion
    -   Stopword removal: Generally helpful to focus on
        sentiment-bearing words
    -   Negation handling: Critical to preserve negations like “not”
        that reverse sentiment
    -   Lemmatization: Preferable to stemming for sentiment analysis as
        it preserves meaning better
4.  Task-dependent preprocessing:
    -   Topic modeling: Aggressive stopword removal and lemmatization to
        focus on content words
    -   Author identification: May preserve capitalization, punctuation
        patterns as stylistic markers
    -   Information retrieval: Stemming might be sufficient for keyword
        matching
    -   Sentiment analysis: Preserve negations, consider lemmatization,
        possibly keep emoticons
    -   Text summarization: Minimal preprocessing to preserve sentence
        structure

</details>

## 3. Text Vectorization: Converting Text to Numbers

To analyze text using machine learning, we need to convert it to
numerical features. Let’s explore common approaches.

### 3.1 Bag of Words with CountVectorizer

``` python
# Create a CountVectorizer
count_vectorizer = CountVectorizer()

# Fit and transform the preprocessed texts
X_counts = count_vectorizer.fit_transform(preprocessed_texts)

# Convert to DataFrame for better visualization
count_df = pd.DataFrame(
    X_counts.toarray(),
    columns=count_vectorizer.get_feature_names_out()
)

print("Bag of Words representation:")
print(count_df)

# Get vocabulary and word counts
vocab = count_vectorizer.get_feature_names_out()
print(f"\nVocabulary size: {len(vocab)}")
print(f"First 10 words in vocabulary: {vocab[:10]}")

# Visualize word frequencies across all documents
word_freq = count_df.sum().sort_values(ascending=False)
top_words = word_freq.head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_words.values, y=top_words.index)
plt.title('Top 15 Words by Frequency')
plt.xlabel('Frequency')
plt.tight_layout()
plt.show()
```

### 3.2 TF-IDF Vectorization

TF-IDF (Term Frequency-Inverse Document Frequency) gives more weight to
words that are important in a document but not common across all
documents.

``` python
# Create a TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer()

# Fit and transform the preprocessed texts
X_tfidf = tfidf_vectorizer.fit_transform(preprocessed_texts)

# Convert to DataFrame for better visualization
tfidf_df = pd.DataFrame(
    X_tfidf.toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
)

print("TF-IDF representation:")
print(tfidf_df)

# Compare the representations for the first document
print("\nComparison for the first document:")
first_doc_comparison = pd.DataFrame({
    'Word': count_vectorizer.get_feature_names_out(),
    'Count': X_counts.toarray()[0],
    'TF-IDF': X_tfidf.toarray()[0]
}).sort_values('TF-IDF', ascending=False)

print(first_doc_comparison.head(10))

# Visualize the differences
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
top_count_words = first_doc_comparison.sort_values('Count', ascending=False).head(10)
sns.barplot(x='Count', y='Word', data=top_count_words)
plt.title('Top Words by Count')

plt.subplot(1, 2, 2)
top_tfidf_words = first_doc_comparison.sort_values('TF-IDF', ascending=False).head(10)
sns.barplot(x='TF-IDF', y='Word', data=top_tfidf_words)
plt.title('Top Words by TF-IDF')

plt.tight_layout()
plt.show()
```

### 3.3 Word Clouds for Visualization

Word clouds are a popular way to visualize the importance of words in a
corpus.

``` python
# Create a word cloud for all the preprocessed texts
all_text = ' '.join(preprocessed_texts)
wordcloud = WordCloud(width=800, height=400, background_color='white',
                      max_words=100, contour_width=3, contour_color='steelblue')
wordcloud.generate(all_text)

plt.figure(figsize=(10, 6))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Preprocessed Texts')
plt.show()

# Create word clouds for positive and negative texts
positive_text = ' '.join([preprocessed_texts[0], preprocessed_texts[3]])
negative_text = preprocessed_texts[1]

positive_cloud = WordCloud(width=400, height=300, background_color='white',
                          max_words=50, contour_width=3, contour_color='steelblue')
positive_cloud.generate(positive_text)

negative_cloud = WordCloud(width=400, height=300, background_color='white',
                          max_words=50, contour_width=3, contour_color='crimson')
negative_cloud.generate(negative_text)

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.imshow(positive_cloud, interpolation='bilinear')
plt.axis('off')
plt.title('Positive Sentiment')

plt.subplot(1, 2, 2)
plt.imshow(negative_cloud, interpolation='bilinear')
plt.axis('off')
plt.title('Negative Sentiment')

plt.tight_layout()
plt.show()
```

## 4. Sentiment Classification

Now let’s apply what we’ve learned to build a sentiment classification
model. We’ll use the newsgroups dataset loaded through our utility
function.

``` python
# Load the newsgroups dataset
texts, labels = load_newsgroup_sentiment()

print(f"Dataset loaded: {len(texts)} documents with {len(np.unique(labels))} classes")
print(f"Class distribution: {pd.Series(labels).value_counts()}")

# Print a few examples
for i in range(3):
    print(f"\nDocument {i+1} (Class: {labels[i]}):")
    # Print just the first 150 characters
    print(f"{texts[i][:150]}...")
```

### 4.1 Text Preprocessing Pipeline

Let’s build a preprocessing pipeline for our newsgroup data.

``` python
# Define a more comprehensive preprocessing function
def preprocess_newsgroup(text):
    """Preprocess newsgroup text."""
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+', '', text)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    
    # Remove punctuation
    text = re.sub(f'[{string.punctuation}]', ' ', text)
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [token for token in tokens if token not in stop_words]
    
    # Lemmatize
    lemmatizer = WordNetLemmatizer()
    tokens = [lemmatizer.lemmatize(token) for token in tokens]
    
    # Remove short tokens
    tokens = [token for token in tokens if len(token) > 2]
    
    # Join tokens back into a string
    processed_text = ' '.join(tokens)
    
    return processed_text

# Preprocess a subset of the data for demonstration
num_samples = min(1000, len(texts))
sample_indices = np.random.choice(len(texts), num_samples, replace=False)
sample_texts = [texts[i] for i in sample_indices]
sample_labels = [labels[i] for i in sample_indices]

# Apply preprocessing
preprocessed_sample = [preprocess_newsgroup(text) for text in sample_texts]
print(f"Preprocessed {len(preprocessed_sample)} documents")

# Print a sample
for i in range(2):
    print(f"\nOriginal: {sample_texts[i][:100]}...")
    print(f"Preprocessed: {preprocessed_sample[i][:100]}...")
```

### 4.2 Building a Sentiment Classifier

Let’s build a classification pipeline using TF-IDF and logistic
regression.

``` python
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    preprocessed_sample, sample_labels, test_size=0.2, random_state=42
)

# Create a pipeline with TF-IDF and logistic regression
sentiment_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('classifier', LogisticRegression(random_state=42))
])

# Train the model
sentiment_pipeline.fit(X_train, y_train)

# Make predictions
y_pred = sentiment_pipeline.predict(X_test)

# Evaluate the model
print("Classification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
conf_matrix = confusion_matrix(y_test, y_pred)
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()
```

### 4.3 Feature Importance Analysis

Let’s examine which words are most important for sentiment
classification.

``` python
# Get feature names and coefficients
tfidf = sentiment_pipeline.named_steps['tfidf']
classifier = sentiment_pipeline.named_steps['classifier']

# Get feature names
feature_names = tfidf.get_feature_names_out()

# Get coefficients
coefficients = classifier.coef_[0]

# Create a DataFrame of words and their coefficients
word_coef = pd.DataFrame({
    'word': feature_names,
    'coefficient': coefficients
})

# Sort by absolute coefficient value
word_coef['abs_coef'] = word_coef['coefficient'].abs()
word_coef = word_coef.sort_values('abs_coef', ascending=False)

# Get top positive and negative words
top_positive = word_coef[word_coef['coefficient'] > 0].head(15)
top_negative = word_coef[word_coef['coefficient'] < 0].head(15)

# Visualize
plt.figure(figsize=(12, 10))

plt.subplot(2, 1, 1)
sns.barplot(x='coefficient', y='word', data=top_positive)
plt.title('Top 15 Words Associated with Positive Sentiment')
plt.xlabel('Coefficient')

plt.subplot(2, 1, 2)
sns.barplot(x='coefficient', y='word', data=top_negative)
plt.title('Top 15 Words Associated with Negative Sentiment')
plt.xlabel('Coefficient')

plt.tight_layout()
plt.show()
```

### Interactive Exercise 2: Model Interpretation and Improvement

🔍 **Analyze the sentiment classification results and consider these
questions:**

1.  What words are strongly associated with positive and negative
    sentiment?
2.  Are there any surprising words in the top features? Why might they
    be important?
3.  How might you improve the model’s performance?
4.  What additional preprocessing steps or feature engineering could
    help?

<details>
<summary>
Click for answers
</summary>

1.  Words associated with sentiment:
    -   Positive sentiment: Words like “good”, “great”, “excellent”,
        “interesting”, “best”, etc.
    -   Negative sentiment: Words like “bad”, “worst”, “terrible”,
        “boring”, “waste”, etc.
    -   Domain-specific words may also appear (e.g., “hockey” might be
        positive in sports newsgroups)
2.  Surprising features:
    -   Names or specific topics might appear if they’re consistently
        associated with one sentiment
    -   Technical terms might appear because of the newsgroup domains
    -   Some words might appear due to sampling bias or artifacts in the
        dataset
    -   Certain neutral words might gain importance if they frequently
        co-occur with sentiment-bearing words
3.  Model improvement strategies:
    -   Use a larger training dataset
    -   Try different classification algorithms (SVM, Random Forest,
        etc.)
    -   Implement ensemble methods combining multiple classifiers
    -   Add n-grams to capture phrases and word sequences
    -   Fine-tune hyperparameters via grid search
    -   Consider deep learning approaches like LSTM or transformers
4.  Additional preprocessing and feature engineering:
    -   Handle negations specially (e.g., “not good” → “not_good”)
    -   Include bigrams and trigrams to capture phrases
    -   Use word embeddings (Word2Vec, GloVe, etc.) instead of TF-IDF
    -   Add part-of-speech tags as features
    -   Include sentiment lexicon scores as features
    -   Consider document length or other structural features
    -   Implement specific handling for emoticons and abbreviations

</details>

## 5. Advanced Text Classification Techniques

Let’s explore some more advanced techniques for text classification.

### 5.1 N-grams for Capturing Phrases

``` python
# Create a TF-IDF vectorizer with unigrams and bigrams
ngram_vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)

# Create a pipeline with n-grams and logistic regression
ngram_pipeline = Pipeline([
    ('tfidf', ngram_vectorizer),
    ('classifier', LogisticRegression(random_state=42))
])

# Train and evaluate
ngram_pipeline.fit(X_train, y_train)
y_pred_ngram = ngram_pipeline.predict(X_test)

print("N-gram Model Classification Report:")
print(classification_report(y_test, y_pred_ngram))

# Compare with previous model
print("\nAccuracy comparison:")
print(f"Unigram model: {accuracy_score(y_test, y_pred):.4f}")
print(f"N-gram model: {accuracy_score(y_test, y_pred_ngram):.4f}")

# Extract top n-gram features
ngram_tfidf = ngram_pipeline.named_steps['tfidf']
ngram_classifier = ngram_pipeline.named_steps['classifier']

# Get feature names and coefficients
ngram_features = ngram_tfidf.get_feature_names_out()
ngram_coefficients = ngram_classifier.coef_[0]

# Create DataFrame of n-grams and coefficients
ngram_coef = pd.DataFrame({
    'ngram': ngram_features,
    'coefficient': ngram_coefficients
})

# Sort by absolute coefficient value
ngram_coef['abs_coef'] = ngram_coef['coefficient'].abs()
ngram_coef = ngram_coef.sort_values('abs_coef', ascending=False)

# Filter for bigrams only
bigrams = ngram_coef[ngram_coef['ngram'].str.contains(' ')]

# Get top positive and negative bigrams
top_positive_bigrams = bigrams[bigrams['coefficient'] > 0].head(15)
top_negative_bigrams = bigrams[bigrams['coefficient'] < 0].head(15)

# Visualize
plt.figure(figsize=(12, 10))

plt.subplot(2, 1, 1)
sns.barplot(x='coefficient', y='ngram', data=top_positive_bigrams)
plt.title('Top 15 Bigrams Associated with Positive Sentiment')
plt.xlabel('Coefficient')

plt.subplot(2, 1, 2)
sns.barplot(x='coefficient', y='ngram', data=top_negative_bigrams)
plt.title('Top 15 Bigrams Associated with Negative Sentiment')
plt.xlabel('Coefficient')

plt.tight_layout()
plt.show()
```

### 5.2 Random Forest for Text Classification

``` python
# Create a pipeline with TF-IDF and Random Forest
rf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
])

# Train and evaluate
rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))

# Compare all models
print("\nAccuracy comparison:")
print(f"Logistic Regression (unigrams): {accuracy_score(y_test, y_pred):.4f}")
print(f"Logistic Regression (n-grams): {accuracy_score(y_test, y_pred_ngram):.4f}")
print(f"Random Forest: {accuracy_score(y_test, y_pred_rf):.4f}")

# Feature importance from Random Forest
rf_tfidf = rf_pipeline.named_steps['tfidf']
rf_classifier = rf_pipeline.named_steps['classifier']

# Get feature names and importance
rf_features = rf_tfidf.get_feature_names_out()
rf_importance = rf_classifier.feature_importances_

# Create DataFrame of features and importance
rf_feature_imp = pd.DataFrame({
    'word': rf_features,
    'importance': rf_importance
})

# Sort by importance
rf_feature_imp = rf_feature_imp.sort_values('importance', ascending=False)

# Visualize top features
plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='word', data=rf_feature_imp.head(20))
plt.title('Top 20 Features by Random Forest Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()
```

## 6. Handling Imbalanced Text Data

Often in sentiment analysis, classes are imbalanced. Let’s explore
techniques to handle this.

``` python
# Create an imbalanced dataset for demonstration
# Select more negative examples than positive
pos_indices = np.where(np.array(sample_labels) == 1)[0]
neg_indices = np.where(np.array(sample_labels) == 0)[0]

# Use fewer positive examples
imbalance_ratio = 0.3  # 30% positive, 70% negative
n_pos = int(len(pos_indices) * imbalance_ratio)
n_neg = len(neg_indices)

# Randomly select positive examples
np.random.seed(42)
selected_pos = np.random.choice(pos_indices, n_pos, replace=False)
selected_indices = np.concatenate([selected_pos, neg_indices])

# Create imbalanced dataset
X_imbalanced = [preprocessed_sample[i] for i in selected_indices]
y_imbalanced = [sample_labels[i] for i in selected_indices]

print(f"Imbalanced dataset: {len(X_imbalanced)} documents")
print(f"Class distribution: {pd.Series(y_imbalanced).value_counts()}")

# Split the imbalanced data
X_train_imb, X_test_imb, y_train_imb, y_test_imb = train_test_split(
    X_imbalanced, y_imbalanced, test_size=0.2, random_state=42
)

# Train a basic model on imbalanced data
imb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('classifier', LogisticRegression(random_state=42))
])

imb_pipeline.fit(X_train_imb, y_train_imb)
y_pred_imb = imb_pipeline.predict(X_test_imb)

print("\nBasic model on imbalanced data:")
print(classification_report(y_test_imb, y_pred_imb))

# Train a model with class weighting
weighted_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000)),
    ('classifier', LogisticRegression(class_weight='balanced', random_state=42))
])

weighted_pipeline.fit(X_train_imb, y_train_imb)
y_pred_weighted = weighted_pipeline.predict(X_test_imb)

print("\nWeighted model on imbalanced data:")
print(classification_report(y_test_imb, y_pred_weighted))
```

## 7. Applying to Real-World Text Data

Now let’s apply what we’ve learned to a real-world text dataset. We’ll
use a sample of movie reviews to predict sentiment.

``` python
# Download the IMDB dataset
from sklearn.datasets import load_files
import os
import urllib.request
import tarfile

# Check if the data directory already exists
if not os.path.exists('aclImdb'):
    # Download the dataset
    url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
    urllib.request.urlretrieve(url, 'aclImdb_v1.tar.gz')
    
    # Extract the dataset
    with tarfile.open('aclImdb_v1.tar.gz', 'r:gz') as tar:
        tar.extractall()
    
    # Delete the tar file
    os.remove('aclImdb_v1.tar.gz')
    
    print("Dataset downloaded and extracted.")
else:
    print("Dataset already exists.")

# Let's use a subset of the data for this tutorial
def load_imdb_subset(max_samples=1000):
    """Load a subset of the IMDB dataset."""
    # Load positive reviews
    pos_dir = 'aclImdb/train/pos'
    neg_dir = 'aclImdb/train/neg'
    
    # Get file lists
    pos_files = os.listdir(pos_dir)[:max_samples//2]
    neg_files = os.listdir(neg_dir)[:max_samples//2]
    
    # Read files
    reviews = []
    labels = []
    
    for filename in pos_files:
        if filename.endswith('.txt'):
            with open(os.path.join(pos_dir, filename), 'r', encoding='utf-8') as f:
                reviews.append(f.read())
                labels.append(1)  # Positive
    
    for filename in neg_files:
        if filename.endswith('.txt'):
            with open(os.path.join(neg_dir, filename), 'r', encoding='utf-8') as f:
                reviews.append(f.read())
                labels.append(0)  # Negative
    
    return reviews, labels

# Load a subset of the IMDB dataset
imdb_texts, imdb_labels = load_imdb_subset(max_samples=2000)

print(f"Loaded {len(imdb_texts)} movie reviews")
print(f"Class distribution: {pd.Series(imdb_labels).value_counts()}")

# Print a few examples
for i in range(2):
    print(f"\nReview {i+1} (Label: {'Positive' if imdb_labels[i] == 1 else 'Negative'}):")
    # Print just the first 200 characters
    print(f"{imdb_texts[i][:200]}...")
```

### 7.1 Preprocessing the Movie Reviews

``` python
# Preprocess the movie reviews
print("Preprocessing reviews...")
preprocessed_reviews = [preprocess_newsgroup(text) for text in imdb_texts]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    preprocessed_reviews, imdb_labels, test_size=0.2, random_state=42
)

print(f"Training set: {len(X_train)} reviews")
print(f"Test set: {len(X_test)} reviews")
```

### 7.2 Building and Evaluating Models

``` python
# Create pipelines for different models
models = {
    'TF-IDF + Logistic Regression': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000)),
        ('classifier', LogisticRegression(random_state=42))
    ]),
    'TF-IDF + N-grams + Logistic Regression': Pipeline([
        ('tfidf', TfidfVectorizer(ngram_range=(1, 2), max_features=5000)),
        ('classifier', LogisticRegression(random_state=42))
    ]),
    'TF-IDF + Random Forest': Pipeline([
        ('tfidf', TfidfVectorizer(max_features=5000)),
        ('classifier', RandomForestClassifier(n_estimators=100, random_state=42))
    ])
}

# Train and evaluate each model
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    
    results[name] = {
        'accuracy': accuracy,
        'report': report,
        'predictions': y_pred,
        'model': model
    }
    
    print(f"Accuracy: {accuracy:.4f}")
    print("Classification Report:")
    print(report)

# Compare model accuracies
accuracies = [result['accuracy'] for result in results.values()]
model_names = list(results.keys())

plt.figure(figsize=(10, 6))
sns.barplot(x=accuracies, y=model_names)
plt.xlabel('Accuracy')
plt.title('Model Comparison')
plt.tight_layout()
plt.show()
```

### 7.3 Error Analysis

Let’s examine some of the misclassified reviews to better understand our
model’s weaknesses.

``` python
# Use the best model for error analysis
best_model_name = model_names[np.argmax(accuracies)]
best_model = results[best_model_name]['model']
best_predictions = results[best_model_name]['predictions']

print(f"Error analysis for {best_model_name}:")

# Find misclassified examples
misclassified_indices = np.where(best_predictions != np.array(y_test))[0]
print(f"Number of misclassified reviews: {len(misclassified_indices)}")

# Look at a few misclassified examples
for i in range(min(5, len(misclassified_indices))):
    idx = misclassified_indices[i]
    true_label = 'Positive' if y_test[idx] == 1 else 'Negative'
    pred_label = 'Positive' if best_predictions[idx] == 1 else 'Negative'
    
    print(f"\nMisclassified Review {i+1}:")
    print(f"True Label: {true_label}, Predicted: {pred_label}")
    # Get original text
    original_idx = len(X_train) + idx
    print(f"Review: {imdb_texts[original_idx][:200]}...")
    
    # Try to understand why it was misclassified
    # Get the TF-IDF representation
    tfidf = best_model.named_steps['tfidf']
    X_tfidf = tfidf.transform([X_test[idx]])
    
    # Get feature names
    feature_names = tfidf.get_feature_names_out()
    
    # Get the top features for this document
    feature_importance = np.argsort(X_tfidf.toarray()[0])[::-1]
    top_features = [feature_names[i] for i in feature_importance[:10]]
    
    print(f"Top features: {', '.join(top_features)}")
```

### Interactive Exercise 3: Improving Sentiment Analysis

🔍 **Based on your error analysis, consider these questions:**

1.  What patterns do you notice in misclassified reviews?
2.  How could you modify your preprocessing to address these issues?
3.  What additional features might help improve the model?
4.  How might you handle complex sentiment like sarcasm or mixed
    opinions?

<details>
<summary>
Click for answers
</summary>

1.  Common patterns in misclassified reviews:
    -   Sarcasm and irony where sentiment is inverted
    -   Mixed reviews with both positive and negative aspects
    -   Subtle or nuanced expressions that don’t use obvious sentiment
        words
    -   Reviews where the overall sentiment differs from the majority of
        individual sentences
    -   Cultural references or idioms that have implicit sentiment
2.  Preprocessing improvements:
    -   Preserve negations and context (e.g., “not good” → “not_good”)
    -   Add special handling for sarcasm markers like quotes or
        exclamation marks
    -   Preserve emoticons and emoji that indicate sentiment
    -   Implement more sophisticated sentence parsing
    -   Add domain-specific knowledge (movie terminology)
3.  Additional helpful features:
    -   Sentiment scores from lexicons like VADER or TextBlob
    -   Review length (very short or very long reviews may have
        different patterns)
    -   Punctuation features (e.g., number of exclamation marks)
    -   Part-of-speech patterns (e.g., ratio of adjectives to nouns)
    -   Sentiment at beginning vs. end of review (conclusions often
        matter more)
    -   Aspect-based sentiment features that capture opinions about
        specific movie elements
4.  Handling complex sentiment:
    -   Use sentence-level analysis before aggregating to document level
    -   Implement aspect-based sentiment analysis to capture nuanced
        opinions
    -   Consider sequence models (RNNs, LSTMs) that can capture context
    -   Use pre-trained language models like BERT that understand
        context better
    -   Add features specifically designed to detect sarcasm
    -   Consider ensemble methods that combine different approaches

</details>

## 8. Advanced Topic: Word Embeddings

Word embeddings are dense vector representations of words that capture
semantic meaning. Let’s explore a simple implementation.

``` python
# For demonstration, we'll use a pre-trained Word2Vec model
from gensim.models import KeyedVectors
import numpy as np

# Download a pre-trained Word2Vec model
word2vec_sample_path = 'word2vec_sample.bin'

# Check if the model already exists
if not os.path.exists(word2vec_sample_path):
    # Download the small sample model
    url = "https://s3.amazonaws.com/dl4j-distribution/GoogleNews-vectors-negative300-5000.bin.gz"
    urllib.request.urlretrieve(url, 'word2vec_sample.bin.gz')
    
    # Extract the model
    import gzip
    with gzip.open('word2vec_sample.bin.gz', 'rb') as f_in:
        with open(word2vec_sample_path, 'wb') as f_out:
            f_out.write(f_in.read())
    
    # Delete the compressed file
    os.remove('word2vec_sample.bin.gz')
    
    print("Word2Vec sample model downloaded and extracted.")
else:
    print("Word2Vec sample model already exists.")

# Load the model
word2vec_model = KeyedVectors.load_word2vec_format(word2vec_sample_path, binary=True)

# Explore the model
print("\nExploring Word2Vec model:")
print(f"Vocabulary size: {len(word2vec_model.key_to_index)}")
print(f"Vector dimension: {word2vec_model.vector_size}")

# Find similar words
try:
    similar_words = word2vec_model.most_similar('movie', topn=5)
    print(f"\nWords similar to 'movie': {similar_words}")
    
    similar_words = word2vec_model.most_similar('terrible', topn=5)
    print(f"Words similar to 'terrible': {similar_words}")
    
    # Word analogies
    result = word2vec_model.most_similar(positive=['woman', 'king'], negative=['man'], topn=1)
    print(f"\nking - man + woman = {result}")
    
except KeyError as e:
    print(f"Note: Some words might not be in the vocabulary of this small sample model: {e}")

# Function to create document vectors from word embeddings
def document_vector(doc, model, vector_size=300):
    """Create a document vector by averaging word vectors."""
    # Initialize empty vector
    doc_vector = np.zeros(vector_size)
    
    # Tokenize
    words = doc.split()
    
    # Count words found in model
    word_count = 0
    
    # Sum word vectors
    for word in words:
        if word in model.key_to_index:
            doc_vector += model[word]
            word_count += 1
    
    # Average
    if word_count > 0:
        doc_vector /= word_count
    
    return doc_vector

# Try document vectorization on a few examples
print("\nCreating document vectors:")
for i in range(2):
    doc = X_test[i]
    vec = document_vector(doc, word2vec_model)
    label = 'Positive' if y_test[i] == 1 else 'Negative'
    print(f"Document {i+1} ({label}) - Vector norm: {np.linalg.norm(vec):.4f}")
```

## 9. Practice Exercise: Building Your Own Review Classifier

Now it’s your turn to build a sentiment classifier for a different type
of text data. Let’s use product reviews from Amazon.

``` python
# For this exercise, we'll use a sample of Amazon product reviews
# You can extend this with your own dataset

# Sample Amazon product reviews
amazon_reviews = [
    "This product is amazing! It exceeded all my expectations and works perfectly.",
    "Don't waste your money. It broke after two days and customer service was terrible.",
    "Good value for the price. Not the best quality but it does the job.",
    "I've had this for a year now and it's still working great. Very satisfied.",
    "The product arrived damaged and the return process was a nightmare.",
    "Average product. Nothing special but no major issues either.",
    "This is the best purchase I've made all year! Absolutely love it!",
    "Misleading description. The actual product is much smaller than advertised.",
    "Works as expected. Good price point and fast shipping.",
    "Terrible design. Difficult to use and feels cheaply made."
]

# Labels (1 for positive, 0 for negative)
amazon_labels = [1, 0, 1, 1, 0, 1, 1, 0, 1, 0]

# Your task:
# 1. Preprocess the reviews
# 2. Create a bag-of-words or TF-IDF representation
# 3. Build a classifier
# 4. Evaluate its performance using cross-validation
# 5. Analyze which words are most predictive of positive and negative reviews
```

<details>
<summary>
Click for sample solution
</summary>

``` python
# 1. Preprocess the reviews
preprocessed_amazon = [preprocess_text(review, remove_stopwords=True, lemmatize=True) for review in amazon_reviews]

# 2. Create TF-IDF representation
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_score, cross_val_predict

tfidf = TfidfVectorizer(max_features=100)
X_tfidf = tfidf.fit_transform(preprocessed_amazon)

# 3. Build a classifier
from sklearn.linear_model import LogisticRegression
classifier = LogisticRegression(random_state=42)

# 4. Evaluate with cross-validation
cv_scores = cross_val_score(classifier, X_tfidf, amazon_labels, cv=5)
print(f"Cross-validation scores: {cv_scores}")
print(f"Average accuracy: {cv_scores.mean():.4f}")

# Get predictions for analysis
y_pred = cross_val_predict(classifier, X_tfidf, amazon_labels, cv=5)
print("\nClassification Report:")
print(classification_report(amazon_labels, y_pred))

# 5. Analyze predictive words
# Fit on all data for interpretation
classifier.fit(X_tfidf, amazon_labels)

# Get feature names and coefficients
feature_names = tfidf.get_feature_names_out()
coefficients = classifier.coef_[0]

# Create DataFrame
word_importance = pd.DataFrame({
    'word': feature_names,
    'coefficient': coefficients
})

# Sort by coefficient value
word_importance = word_importance.sort_values('coefficient', ascending=False)

# Top positive and negative words
print("\nTop words for positive reviews:")
print(word_importance.head(10))

print("\nTop words for negative reviews:")
print(word_importance.tail(10))

# Visualize
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
sns.barplot(x='coefficient', y='word', data=word_importance.head(10))
plt.title('Top Words for Positive Reviews')

plt.subplot(1, 2, 2)
sns.barplot(x='coefficient', y='word', data=word_importance.tail(10).sort_values('coefficient'))
plt.title('Top Words for Negative Reviews')

plt.tight_layout()
plt.show()

# Note: With only 10 reviews, cross-validation results will have high variance
# In a real application, you would use a much larger dataset
```

</details>

## 10. Summary and Key Takeaways

In this tutorial, we’ve covered:

1.  **Text preprocessing** - Converting raw text to a clean format
2.  **Text vectorization** - Converting text to numerical features with
    Bag of Words and TF-IDF
3.  **Sentiment classification** - Building models to predict sentiment
4.  **Feature importance** - Identifying words and phrases that drive
    sentiment
5.  **Advanced techniques** - N-grams, class weighting, and word
    embeddings

### Next Steps

In the next tutorial, we’ll explore: - Ensemble methods and model
stacking - Neural networks for sequence data - Cross-validation
techniques

## 11. Additional Resources

-   [NLTK Documentation](https://www.nltk.org/)
-   [Scikit-learn Text Feature Extraction
    Guide](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction)
-   [spaCy: Industrial-Strength NLP](https://spacy.io/)
-   [Hugging Face Transformers](https://huggingface.co/transformers/)
    for state-of-the-art NLP models
-   [Stanford NLP Group](https://nlp.stanford.edu/software/)
-   [Text Classification with NLTK and
    Scikit-learn](https://www.analyticsvidhya.com/blog/2018/04/a-comprehensive-guide-to-understand-and-implement-text-classification-in-python/)
-   [Word2Vec
    Tutorial](https://radimrehurek.com/gensim/models/word2vec.html)
-   [VADER Sentiment
    Analysis](https://github.com/cjhutto/vaderSentiment)

## Appendix: Mathematical Foundations

### A.1 TF-IDF Formula

TF-IDF stands for Term Frequency-Inverse Document Frequency:

-   **Term Frequency (TF)**: The number of times a term appears in a
    document, divided by the total number of terms in the document.

$\text{TF}(t, d) = \frac{\text{count of term } t \text{ in document } d}{\text{total number of terms in document } d}$

-   **Inverse Document Frequency (IDF)**: Measures how important a term
    is by taking the logarithm of the ratio of the total number of
    documents to the number of documents containing the term.

$\text{IDF}(t, D) = \log\frac{\text{total number of documents in corpus } D}{\text{number of documents containing term } t}$

-   **TF-IDF**: The product of TF and IDF.

$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$

### A.2 Word Embeddings

Word embeddings are dense vector representations of words in a
continuous vector space. Unlike one-hot encoding, embeddings place
semantically similar words close to each other in the vector space.

Word2Vec uses two main architectures: - **Continuous Bag of Words
(CBOW)**: Predicts the target word from context words - **Skip-gram**:
Predicts context words from the target word

The training objective for Skip-gram is to maximize:

$J(\theta) = \frac{1}{T} \sum_{t=1}^{T} \sum_{-c \leq j \leq c, j \neq 0} \log p(w_{t+j} | w_t)$

where $c$ is the size of the context window and $p(w_{t+j} | w_t)$ is
modeled using the softmax function:

$p(w_O | w_I) = \frac{\exp(v'_{w_O} \cdot v_{w_I})}{\sum_{w=1}^{W} \exp(v'_{w} \cdot v_{w_I})}$

where $v_{w}$ and $v'_{w}$ are the “input” and “output” vector
representations of word $w$.